In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

In [ ]:
model_id = "meta-llama/Llama-3.2-1B" 


In [2]:
# 2. Load the Tokenizer and Model
print("Loading model and tokenizer on CPU...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

Loading model and tokenizer on CPU...


NameError: name 'AutoTokenizer' is not defined

In [ ]:
# Load model explicitly in 32-bit float or 16-bit float for CPU
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.float32, 
    device_map={"":"cpu"} # Force CPU placement
)

In [ ]:
# 3. Load a Small, Clean Dataset
print("Loading dataset...")
# Using a small subset of the Hugging Face SmolLM corpus for quick demo training
dataset = load_dataset("HuggingFaceTB/smollm-corpus", "cosmopedia-v2", split="train[:500]")


In [ ]:

# 4. Set up LoRA (Parameter Efficient Fine Tuning)
# This trains less than 1% of the model parameters, saving immense memory and time
peft_config = LoraConfig(
    r=8, 
    lora_alpha=16, 
    target_modules=["q_proj", "v_proj"], # Target key attention layers
    lora_dropout=0.05, 
    bias="none", 
    task_type="CAUSAL_LM"
)

In [ ]:

# 5. Define Training Arguments Optimized for CPU
training_args = TrainingArguments(
    output_dir="./slm_finetuned_results",
    per_device_train_batch_size=1,      # Kept at 1 to prevent CPU RAM spikes
    gradient_accumulation_steps=4,     # Simulates a batch size of 4
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=50,                       # Small step count for a fast test run
    optim="adamw_hf",                  # Standard CPU-compatible optimizer
    remove_unused_columns=True,
    use_cpu=True,                       # Explicitly tells HF to ignore GPUs
    report_to="none"                    # Disables external tracking dashboards
)


In [ ]:
# 6. Initialize the Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",         # Adjust this based on your dataset column name
    max_seq_length=512,                # Caps context window to speed up processing
    tokenizer=tokenizer,
    ar

In [ ]:
print("Starting fine-tuning on your Intel CPU...")
trainer.train()


In [ ]:
# 8. Save the Fine-Tuned Adapter Weights
print("Training complete! Saving adapter...")
trainer.model.save_pretrained("./my_saved_slm_adapter")

In [3]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

# 1. Load your model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-14B", # Or your preferred model
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

# 2. Add LoRA adapters (so you only train a fraction of the parameters)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
)

# 3. Configure and initialize the trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=my_dataset, # Replace with your dataset
    dataset_text_field="text",
    max_seq_length=4096,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=10,
        output_dir="outputs",
    ),
)

# 4. Enable/Initiate the training loop
trainer.train()


ModuleNotFoundError: No module named 'unsloth'